## 1. Data Loading
Load raw transaction JSON files from `data/raw/transactions/` and combine into a single DataFrame.

In [1]:
import pandas as pd
import numpy as np
import json
import tabulate
from pathlib import Path
from eth_utils import decode_hex


# Configure paths
RAW_DATA_DIR = Path('../data/raw/transactions') 

# Load all transaction JSON files
all_transactions = []
for tx_file in RAW_DATA_DIR.glob('*.json'):
    with open(tx_file) as f:
        transactions = json.load(f)
        all_transactions.extend(transactions)

# Create DataFrame
tx_df = pd.DataFrame(all_transactions)
print(f"Loaded {len(tx_df):,} transactions")

Loaded 22,189 transactions


## 2. Data Cleaning
Convert hexadecimal values to integers and handle missing data.

In [2]:
def safe_hex_to_int_or_float(value_to_convert):
    """
    Safely converts a hexadecimal string (e.g., '0x...') to an integer or float.
    Returns the number if already numeric (int, float).
    Returns np.nan for unparseable strings or other non-convertible types.
    """
    if isinstance(value_to_convert, (int, float)): # If already numeric, return as is
        return value_to_convert
    if isinstance(value_to_convert, str):
        if value_to_convert.startswith('0x'):
            try:
                return int(value_to_convert, 16)
            except ValueError: # Malformed hex string
                return np.nan
        else: # String, but not starting with '0x' - try to parse as decimal int or float
            try:
                # Attempt to convert to int first, then float if int fails
                return int(value_to_convert)
            except ValueError:
                try:
                    return float(value_to_convert)
                except ValueError:
                    return np.nan # Cannot parse as any number
    return np.nan # For other types like None, etc.

# Apply the robust conversion function to numeric fields
# For 'value', 'gas', 'gasPrice', 'gasUsed', 'blockNumber', 'timeStamp', 'nonce', 'transactionIndex', 'confirmations'
# We expect these to be numbers or hex strings representing numbers.
numeric_cols = ['value', 'gas', 'gasPrice', 'gasUsed', 'blockNumber', 'timeStamp', 'nonce', 'transactionIndex', 'confirmations']
for col in numeric_cols:
    if col in tx_df.columns:
        print(f"Converting column: {col}")
        tx_df[col] = tx_df[col].apply(safe_hex_to_int_or_float)
        # Ensure they are treated as numeric by pandas, coercing errors introduced by apply if any (should be NaN now)
        tx_df[col] = pd.to_numeric(tx_df[col], errors='coerce')
    else:
        print(f"Warning: Column {col} not found in DataFrame for numeric conversion.")


# Specific handling for 'value': fill NaNs with 0 after conversion, as value is critical
if 'value' in tx_df.columns:
    tx_df['value'] = tx_df['value'].fillna(0).astype(np.int64) # Or float64 if preferred and NaNs were kept

# Handle missing 'input' data (often for simple ETH transfers)
if 'input' in tx_df.columns:
    tx_df['input'] = tx_df['input'].fillna('0x')
else:
    print("Warning: 'input' column not found. Creating it with default '0x'.")
    tx_df['input'] = '0x'


# Verify dtypes after conversion
print("\nData types after robust cleaning in Section 2:")
print(tx_df[numeric_cols + ['input']].dtypes.to_string())

Converting column: value
Converting column: gas
Converting column: gasPrice
Converting column: gasUsed
Converting column: blockNumber
Converting column: timeStamp
Converting column: nonce
Converting column: transactionIndex
Converting column: confirmations

Data types after robust cleaning in Section 2:
value                int64
gas                  int64
gasPrice             int64
gasUsed              int64
blockNumber          int64
timeStamp            int64
nonce                int64
transactionIndex     int64
confirmations        int64
input               object


## 3. Feature Engineering
Create meaningful features for machine learning.


In [3]:
# Temporal features
tx_df['timestamp'] = pd.to_datetime(tx_df['timeStamp'], unit='s')
tx_df['hour_of_day'] = tx_df['timestamp'].dt.hour

# Transaction relationships
tx_df['from_to_pair'] = tx_df['from'] + '_' + tx_df['to']

# Gas efficiency ratio
tx_df['gas_efficiency'] = tx_df['gasUsed'] / tx_df['gas']
tx_df['gas_efficiency'] = tx_df['gas_efficiency'].replace([np.inf, -np.inf], 0)

## 4. Smart Contract Interaction Analysis
Decode function signatures from input data.

In [4]:
def extract_function_signature(input_data: str):
    # ... (code de la fonction comme défini précédemment) ...
    if not isinstance(input_data, str) or not input_data.startswith('0x'):
        return None
    if len(input_data) >= 10:
        hex_signature_part = input_data[2:10]
        try:
            return decode_hex(hex_signature_part)
        except Exception:
            return None
    return None

tx_df['function_sig_bytes'] = tx_df['input'].apply(extract_function_signature)

# ---> CETTE LIGNE EST CRUCIALE <---
tx_df['function_sig_hex'] = tx_df['function_sig_bytes'].apply(
    lambda b: '0x' + b.hex() if isinstance(b, bytes) else None
)

## 5. Data Validation
Ensure data quality before saving processed dataset.

In [5]:
# Check for missing values across all columns
print("Missing values per column after all cleaning:")
missing_values = tx_df.isna().sum()
print(missing_values[missing_values > 0]) # Print only columns with missing values

# Verify value ranges, especially for the 'value' column (converted to ETH)
# Ensure 'value' column exists and is numeric before this operation
if 'value' in tx_df.columns and pd.api.types.is_numeric_dtype(tx_df['value']):
    print("\nValue statistics (in ETH):")
    # Convert Wei to ETH for describe(): 1 ETH = 10^18 Wei
    value_in_eth = tx_df['value'] / 1e18
    print(value_in_eth.describe())
else:
    print("\n'value' column is not numeric or does not exist. Cannot calculate ETH statistics.")
    if 'value' in tx_df.columns:
        print(f"Debug: 'value' column dtype is {tx_df['value'].dtype}")

# Example: Display some rows to manually inspect data consistency
print("\nSample of cleaned data (first 5 rows):")
print(tx_df.head().to_markdown(index=False)) # Using to_markdown for better notebook display

Missing values per column after all cleaning:
function_sig_bytes    16618
function_sig_hex      16618
dtype: int64

Value statistics (in ETH):
count    22189.000000
mean        -0.056612
std          3.225171
min         -9.223372
25%          0.000000
50%          0.090000
75%          0.960000
max          9.171600
Name: value, dtype: float64

Sample of cleaned data (first 5 rows):
|   blockNumber | blockHash                                                          |   timeStamp | hash                                                               |   nonce |   transactionIndex | from                                       | to                                         |               value |   gas |    gasPrice | input   | methodId   | functionName   | contractAddress   |   cumulativeGasUsed |   txreceipt_status |   gasUsed |   confirmations |   isError | timestamp           |   hour_of_day | from_to_pair                                                                          |   gas_e

## 6. Save Processed Data
Export cleaned dataset for model training.

In [6]:
# Define the output path for the processed data
# Adjust the path if your notebook is in a different subdirectory relative to the 'data' folder
# Assuming the notebook is in 'EthMalViz/notebooks/'
PROCESSED_DIR = Path('../data/processed/') # Path relative to the notebook's location
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) # Ensure the directory exists

PROCESSED_FILE_PATH = PROCESSED_DIR / 'transactions_cleaned.parquet'

try:
    # Save the DataFrame to a Parquet file using the pyarrow engine by default
    # Pandas will automatically try pyarrow first if it's installed.
    # You can also be explicit: engine='pyarrow'
    print(f"Attempting to save DataFrame to: {PROCESSED_FILE_PATH}")
    print(f"DataFrame shape: {tx_df.shape}")
    print(f"DataFrame dtypes sample:\n{tx_df.dtypes.head().to_string()}") # Show a sample of dtypes

    tx_df.to_parquet(PROCESSED_FILE_PATH, index=False, engine='pyarrow') # Explicitly using pyarrow
    
    print(f"\nSuccessfully saved processed data to: {PROCESSED_FILE_PATH}")
    print(f"File size: {PROCESSED_FILE_PATH.stat().st_size / (1024*1024):.2f} MB")

except ImportError as e:
    print(f"ImportError: {e}. It seems the Parquet engine (pyarrow or fastparquet) is still not found or correctly installed.")
    print("Please ensure you have run 'pip install pyarrow' in your 'cyber-env-py312' environment and restarted the Jupyter kernel.")
except Exception as e:
    print(f"An unexpected error occurred while saving to Parquet: {e}")
    print("DataFrame info that was attempted to be saved:")
    tx_df.info() # Print df info for debugging if other errors occur

Attempting to save DataFrame to: ../data/processed/transactions_cleaned.parquet
DataFrame shape: (22189, 26)
DataFrame dtypes sample:
blockNumber     int64
blockHash      object
timeStamp       int64
hash           object
nonce           int64

Successfully saved processed data to: ../data/processed/transactions_cleaned.parquet
File size: 5.01 MB
